# 实验二：基于机器学习的人类编写代码的代码审查

## 步骤一：从 lab1 原始数据中筛选人类编写的 PR

根据 lab1 中 `feature_extractor.py` 的 AI 检测逻辑，筛选出人类编写的 PR，保存到本地。

## 准备工作

### 安装依赖

In [1]:
%pip install -r ../requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


---
## 步骤一：筛选人类编写的 PR

**筛选逻辑**（复用 lab1 的 `detect_ai_generated_code`）：
1. **Bot 作者排除**：作者名含 `bot`、`dependabot` 等 → 既不算人类也不算 AI
2. **关键词匹配**：在 commit message / PR body / title / labels 中匹配 AI 关键词（copilot、chatgpt、claude 等）
3. **启发式规则**：单文件大新增 + 极短 commit / PR body 模板化

满足以上任一条件的 PR 标记为 AI 生成，其余为人类编写。

In [2]:
from pr_filter import PRFilter

filter_obj = PRFilter()
summary = filter_obj.run_all()

开始筛选人类编写的 PR...

处理 kubernetes/kubernetes...
  总计: 300, 人类: 295, AI: 5, Bot: 0

处理 pytorch/pytorch...
  总计: 300, 人类: 210, AI: 84, Bot: 6

处理 tensorflow/tensorflow...
  总计: 300, 人类: 11, AI: 3, Bot: 286

处理 microsoft/vscode...
  总计: 300, 人类: 104, AI: 183, Bot: 13

处理 langchain-ai/langchain...
  总计: 300, 人类: 135, AI: 69, Bot: 96

筛选完成!
  总 PR 数: 1500
  人类编写: 755
  AI 生成:  344
  Bot 提交: 401


## 筛选结果汇总

In [3]:
import pandas as pd

df_stats = pd.DataFrame(summary["stats"])
df_stats

,repo,total,human,non_human,bot
0,kubernetes/kubernetes,300,295,5,0
1,pytorch/pytorch,300,210,84,6
2,tensorflow/tensorflow,300,11,3,286
3,microsoft/vscode,300,104,183,13
4,langchain-ai/langchain,300,135,69,96


In [4]:
print(f"总 PR 数:    {summary['total_all']}")
print(f"人类编写:    {summary['total_human']}")
print(f"AI 生成:     {summary['total_non_human']}")
print(f"Bot 提交:    {summary['total_bot']}")
print(f"人类占比:    {summary['total_human'] / summary['total_all'] * 100:.1f}%")

总 PR 数:    1500
人类编写:    755
AI 生成:     344
Bot 提交:    401
人类占比:    50.3%


## 保存人类 PR 汇总 CSV

In [5]:
filter_obj.save_filtered_csv()

人类 PR 汇总已保存到: f:\学习资料\智能软件工程实践\lab2\data\filtered_human_prs.csv (共 755 条)


## 验证输出文件

In [6]:
import os
import config

print("人类 PR 数据文件:")
for f in os.listdir(config.HUMAN_DIR):
    filepath = os.path.join(config.HUMAN_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

print("\nAI 生成 PR 数据文件:")
for f in os.listdir(config.NON_HUMAN_DIR):
    filepath = os.path.join(config.NON_HUMAN_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

人类 PR 数据文件:
  kubernetes_kubernetes_pulls.json (17167.0 KB)
  langchain-ai_langchain_pulls.json (1272.8 KB)
  microsoft_vscode_pulls.json (2395.6 KB)
  pytorch_pytorch_pulls.json (4865.7 KB)
  tensorflow_tensorflow_pulls.json (126.9 KB)

AI 生成 PR 数据文件:
  kubernetes_kubernetes_pulls.json (263.5 KB)
  langchain-ai_langchain_pulls.json (646.5 KB)
  microsoft_vscode_pulls.json (7655.3 KB)
  pytorch_pytorch_pulls.json (19869.8 KB)
  tensorflow_tensorflow_pulls.json (135.8 KB)


In [7]:
csv_path = os.path.join(config.DATA_DIR, "filtered_human_prs.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"filtered_human_prs.csv: {df.shape[0]} 行, {df.shape[1]} 列")
    print(f"\n各仓库人类 PR 数量:")
    print(df.groupby(["repo_owner", "repo_name"]).size())
    print(f"\n前 5 条预览:")
    display(df.head())

filtered_human_prs.csv: 755 行, 12 列

各仓库人类 PR 数量:
repo_owner    repo_name 
kubernetes    kubernetes    295
langchain-ai  langchain     135
microsoft     vscode        104
pytorch       pytorch       210
tensorflow    tensorflow     11
dtype: int64

前 5 条预览:


,pr_id,repo_owner,repo_name,title,author,created_at,additions,deletions,changed_files,is_merged,label_count,labels
0,140072,kubernetes,kubernetes,"cleanup: fix double ""the"" and other comment ty...",shashank-tomar0,2026-06-29T07:14:13+00:00,4,4,4,0,14,"area/kubelet,kind/cleanup,sig/scheduling,sig/n..."
1,140071,kubernetes,kubernetes,[WIP] Return errors from ToleratesTaint for in...,pacoxu,2026-06-29T06:37:34+00:00,166,77,16,0,16,"kind/bug,area/test,area/kubelet,sig/scheduling..."
2,140068,kubernetes,kubernetes,cleanup: remove stale heapster references from...,amigo-nishant,2026-06-29T05:40:44+00:00,4,10,5,0,11,"area/kubelet,kind/cleanup,sig/node,sig/autosca..."
3,140067,kubernetes,kubernetes,flake stabilize HPA reconciliation metric asse...,googs1025,2026-06-29T02:41:15+00:00,13,4,1,0,8,"sig/autoscaling,size/S,release-note-none,sig/a..."
4,140066,kubernetes,kubernetes,Automated cherry pick of #139850: kubelet star...,compumike,2026-06-29T00:03:53+00:00,16,1,2,0,12,"kind/bug,area/kubelet,lgtm,sig/node,release-no..."


---
## 步骤一完成

筛选结果已保存到：
- `lab2/data/raw/human/` — 人类编写的 PR（JSON）
- `lab2/data/raw/non_human/` — AI 生成的 PR（JSON）
- `lab2/data/filtered_human_prs.csv` — 人类 PR 汇总 CSV

---
## 步骤二：提取 PR 修改前后的代码

对步骤一筛选出的每个 PR，下载其修改前（before）和修改后（after）的源代码文件，
过滤出 7 种主流语言（Go、Python、JavaScript、TypeScript、Java、C、C++），保存到本地。

### 获取策略
- **after 代码**：直接从 PR 文件中的 `raw_url` 下载
- **before 代码**：通过 GitHub API 获取 commit 的 parent SHA，再拼接 raw URL 下载

In [8]:
from code_extractor import CodeExtractor

FORCE_CHECK = False  # True=强制重新检查所有文件, False=已下载文件快速跳过
extractor = CodeExtractor()
results = extractor.run_all(force_check=FORCE_CHECK)

快速模式：跳过检查，默认所有文件已下载完成


### 验证提取结果

In [9]:
import os
import config

total_files = 0
total_prs = 0
for repo_entry in os.listdir(config.HUMAN_CODE_DIR):
    repo_path = os.path.join(config.HUMAN_CODE_DIR, repo_entry)
    if not os.path.isdir(repo_path):
        continue
    pr_count = 0
    file_count = 0
    for pr_id in os.listdir(repo_path):
        pr_path = os.path.join(repo_path, pr_id)
        if os.path.isdir(pr_path):
            pr_count += 1
            before_dir = os.path.join(pr_path, "before")
            after_dir = os.path.join(pr_path, "after")
            if os.path.exists(before_dir):
                file_count += len(os.listdir(before_dir))
            if os.path.exists(after_dir):
                file_count += len(os.listdir(after_dir))
    print(f"  {repo_entry}: {pr_count} PRs, {file_count} files")
    total_prs += pr_count
    total_files += file_count

print(f"\n总计: {total_prs} PRs, {total_files} 代码文件")

  kubernetes_kubernetes: 295 PRs, 5357 files
  langchain-ai_langchain: 135 PRs, 552 files
  microsoft_vscode: 104 PRs, 683 files
  pytorch_pytorch: 210 PRs, 1634 files
  tensorflow_tensorflow: 11 PRs, 36 files

总计: 755 PRs, 8262 代码文件


In [10]:
for repo_entry in sorted(os.listdir(config.HUMAN_CODE_DIR)):
    repo_path = os.path.join(config.HUMAN_CODE_DIR, repo_entry)
    if not os.path.isdir(repo_path):
        continue
    pr_ids = sorted(os.listdir(repo_path), key=lambda x: int(x))
    if pr_ids:
        sample_pr = pr_ids[0]
        sample_path = os.path.join(repo_path, sample_pr)
        print(f"\n{repo_entry} / PR {sample_pr}:")
        for sub in ["before", "after"]:
            sub_path = os.path.join(sample_path, sub)
            if os.path.exists(sub_path):
                files = os.listdir(sub_path)[:3]
                print(f"  {sub}/ ({len(os.listdir(sub_path))} files): {files}")


kubernetes_kubernetes / PR 139699:
  before/ (1 files): ['test_integration_garbagecollector_garbage_collector_test.go']
  after/ (1 files): ['test_integration_garbagecollector_garbage_collector_test.go']

langchain-ai_langchain / PR 38166:
  before/ (0 files): []
  after/ (0 files): []

microsoft_vscode / PR 322761:
  before/ (2 files): ['src_vs_workbench_contrib_chat_browser_agentSessions_agentHost_agentHostSessionHandler.ts', 'src_vs_workbench_contrib_chat_test_browser_agentSessions_agentHostChatContribution.test.ts']
  after/ (2 files): ['src_vs_workbench_contrib_chat_browser_agentSessions_agentHost_agentHostSessionHandler.ts', 'src_vs_workbench_contrib_chat_test_browser_agentSessions_agentHostChatContribution.test.ts']

pytorch_pytorch / PR 187995:
  before/ (3 files): ['test_dynamo_test_activation_checkpointing.py', 'test_inductor_test_compiled_autograd.py', 'torch__functorch_partitioners.py']
  after/ (3 files): ['test_dynamo_test_activation_checkpointing.py', 'test_inductor_tes

---
## 步骤二 代码提取完成

代码文件已保存到：
- `lab2/data/code/human/{repo}/{pr_id}/before/` — 修改前代码
- `lab2/data/code/human/{repo}/{pr_id}/after/` — 修改后代码

支持的 7 种语言：Go、Python、JavaScript、TypeScript、Java、C、C++

---
## 步骤二：生成 AST 和 CFG 中间表示

对下载的代码文件，使用 tree-sitter 解析生成 AST（抽象语法树）和 CFG（控制流图），
保存到与 `code/` 同级的 `ast/` 和 `cfg/` 目录下。

输出目录结构：
- `lab2/data/ast/human/{repo}/{pr_id}/before/` — 修改前 AST
- `lab2/data/ast/human/{repo}/{pr_id}/after/` — 修改后 AST
- `lab2/data/cfg/human/{repo}/{pr_id}/before/` — 修改前 CFG
- `lab2/data/cfg/human/{repo}/{pr_id}/after/` — 修改后 CFG

In [11]:
from ast_cfg_generator import ASTCFGGenerator

FORCE_CHECK_AST = False  # True=强制重新生成所有 AST/CFG, False=跳过
generator = ASTCFGGenerator()
generator.run_all(force_check=FORCE_CHECK_AST)

FORCE_CHECK=False, 跳过 AST/CFG 生成


---
## 验证：查看生成结果样例

随机抽查一个 PR 的 AST 和 CFG 文件

In [12]:
import json
import os

def show_sample(base_dir, label):
    print(f"\n{'='*40}")
    print(f"  {label} 样例")
    print(f"{'='*40}")
    for repo_entry in sorted(os.listdir(base_dir))[:3]:
        repo_path = os.path.join(base_dir, repo_entry)
        if not os.path.isdir(repo_path):
            continue
        pr_ids = sorted(os.listdir(repo_path), key=lambda x: int(x))
        if not pr_ids:
            continue
        sample_pr = pr_ids[0]
        for sub in ["before", "after"]:
            sub_path = os.path.join(repo_path, sample_pr, sub)
            if not os.path.exists(sub_path):
                continue
            json_files = [f for f in os.listdir(sub_path) if f.endswith('.json')]
            if json_files:
                sample_file = os.path.join(sub_path, json_files[0])
                with open(sample_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                print(f"\n{repo_entry}/PR {sample_pr}/{sub}/{json_files[0]}:")
                if label == "AST":
                    print(f"  语言: {data.get('language')}")
                    print(f"  AST 节点总数: {data.get('node_count')}")
                    print(f"  控制流节点数: {data.get('control_flow_count')}")
                    sexp = data.get('sexp', '')
                    print(f"  S-expression 前200字符: {sexp[:200]}...")
                else:
                    print(f"  语言: {data.get('language')}")
                    print(f"  函数数量: {data.get('function_count')}")
                    print(f"  CFG 总节点数: {data.get('total_node_count')}")
                    print(f"  CFG 总边数: {data.get('total_edge_count')}")
                break
        break

show_sample(config.HUMAN_AST_DIR, "AST")
show_sample(config.HUMAN_CFG_DIR, "CFG")


  AST 样例

kubernetes_kubernetes/PR 139699/before/test_integration_garbagecollector_garbage_collector_test.json:
  语言: Go
  AST 节点总数: 16822
  控制流节点数: 217
  S-expression 前200字符: (source_file (comment) (package_clause (package_identifier)) (import_declaration (import_spec_list (import_spec path: (interpreted_string_literal)) (import_spec path: (interpreted_string_literal)) (im...

  CFG 样例

kubernetes_kubernetes/PR 139699/before/test_integration_garbagecollector_garbage_collector_test.json:
  语言: Go
  函数数量: 24
  CFG 总节点数: 544
  CFG 总边数: 644


---
## 步骤二 AST/CFG 生成完成

AST 和 CFG 文件已保存到：
- `lab2/data/ast/human/{repo}/{pr_id}/before/` — 修改前 AST
- `lab2/data/ast/human/{repo}/{pr_id}/after/` — 修改后 AST
- `lab2/data/cfg/human/{repo}/{pr_id}/before/` — 修改前 CFG
- `lab2/data/cfg/human/{repo}/{pr_id}/after/` — 修改后 CFG

每个 AST 文件包含：`language`, `node_count`, `control_flow_count`, `sexp`
每个 CFG 文件包含：`language`, `function_count`, `total_node_count`, `total_edge_count`, `functions`（含 graph 结构）

---
## 步骤三：特征提取

从 PR 信息、AST、CFG 中提取特征，采用聚合 + 均值策略，before 和 after 并列。

提取特征：
1. **基础特征 (5维)**：修改文件数、新增行数、删除行数、title长度、body长度
2. **AST特征 (before+after)**：各控制流节点类型的 sum 和 mean
3. **CFG特征 (before+after)**：各节点类型的 sum 和 mean + 结构特征（总节点、总边、函数数、平均节点/函数、循环复杂度）

输出：`data/features/human_features.csv`

In [13]:
from feature_extractor import FeatureExtractor

FORCE_CHECK_FEATURE = False  # True=强制重新提取所有特征, False=跳过
extractor = FeatureExtractor()
csv_path = extractor.run_all(force_check=FORCE_CHECK_FEATURE)


FORCE_CHECK=False, 跳过特征提取


---
## 验证：查看特征 CSV 样例

查看特征文件的前几行和列名

In [14]:
import pandas as pd
import config
import os

csv_path = os.path.join(config.FEATURES_DIR, "human_features.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"特征表形状: {df.shape}")
    print(f"列数: {len(df.columns)}")
    print(f"\n前 5 列:")
    print(df.columns[:5].tolist())
    print(f"\n最后 5 列:")
    print(df.columns[-5:].tolist())
    print(f"\n基础统计:")
    print(df[["file_count", "additions", "deletions", "title_len", "body_len"]].describe())
else:
    print("特征文件不存在，请先运行上一步提取特征。")

特征表形状: (700, 163)
列数: 163

前 5 列:
['pr_id', 'repo', 'file_count', 'additions', 'deletions']

最后 5 列:
['cfg_after_total_edges_mean', 'cfg_after_function_count', 'cfg_after_avg_nodes_per_func', 'cfg_after_avg_edges_per_func', 'cfg_after_cyclomatic_complexity']

基础统计:
       file_count      additions     deletions   title_len      body_len
count  700.000000     700.000000    700.000000  700.000000    700.000000
mean     7.458571     643.171429    106.454286   59.264286   1434.417143
std     21.075306   10981.895190    817.418251   19.603294   1793.989933
min      1.000000       0.000000      0.000000    3.000000      0.000000
25%      2.000000      12.000000      1.000000   47.000000    426.750000
50%      2.000000      50.000000      5.000000   59.000000    921.000000
75%      5.000000     138.250000     25.250000   70.250000   1691.500000
max    248.000000  290137.000000  15373.000000  139.000000  22888.000000


---
## 步骤三 特征提取完成

特征文件已保存到：
- `lab2/data/features/human_features.csv` — 特征表
- `lab2/data/features/feature_columns.txt` — 列名说明

特征维度：基础特征(5) + AST before+after(各~30) + CFG before+after(各~48) ≈ 161维

---
## 步骤四：模型训练

分别训练两个分类模型，预测 PR 是否最终 Merge:
- **Support Vector Machine (SVM)**
- **Random Forest**

输出模型到 `data/models/` 目录

In [15]:
from model_trainer import SVMTrainer, RFTrainer

svm_trainer = SVMTrainer()
svm_trainer.train()

训练 SVM (Support Vector Machine)
数据加载完成: 700 条样本, 161 维特征
  标签分布: merged=191, not_merged=509
训练集: 560, 测试集: 140
Fitting 3 folds for each of 12 candidates, totalling 36 fits
  最佳参数: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
  最佳 CV F1: 0.5324
SVM 模型已保存到: f:\学习资料\智能软件工程实践\lab2\data\models\svm_model.joblib


In [16]:
rf_trainer = RFTrainer()
rf_trainer.train()

训练 Random Forest
数据加载完成: 700 条样本, 161 维特征
  标签分布: merged=191, not_merged=509
训练集: 560, 测试集: 140
Fitting 3 folds for each of 12 candidates, totalling 36 fits
  最佳参数: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
  最佳 CV F1: 0.5565
Random Forest 模型已保存到: f:\学习资料\智能软件工程实践\lab2\data\models\rf_model.joblib


---
## 步骤五：模型评估

测试集上评估模型性能，计算:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

输出:
- 混淆矩阵图
- ROC 曲线图
- Random Forest 特征重要性分析
- 评估报告文本

In [17]:
svm_result = svm_trainer.evaluate()

评估 SVM 模型

SVM 评估结果
  Accuracy:   0.8143
  Precision:  0.7143
  Recall:     0.5263
  F1-score:   0.6061
  ROC-AUC:    0.7758
  混淆矩阵:
    TN=94, FP=8
    FN=18, TP=20


In [18]:
rf_result = rf_trainer.evaluate()

评估 Random Forest 模型

Random Forest 评估结果
  Accuracy:   0.8571
  Precision:  0.8750
  Recall:     0.5526
  F1-score:   0.6774
  ROC-AUC:    0.8777
  混淆矩阵:
    TN=99, FP=3
    FN=17, TP=21

特征重要性 Top 10 (Random Forest)
   1. body_len                                           0.0396
   2. cfg_before_call_expression_mean                    0.0274
   3. cfg_after_call_expression_mean                     0.0242
   4. deletions                                          0.0238
   5. additions                                          0.0199
   6. cfg_after_exit_mean                                0.0182
   7. ast_before_for_statement_mean                      0.0178
   8. cfg_before_return_statement_mean                   0.0174
   9. cfg_before_expression_statement_mean               0.0172
  10. cfg_before_call_expression_sum                     0.0166


In [19]:
from model_trainer import ModelTrainer
trainer = ModelTrainer()
trainer.evaluate_both()

模型对比评估
数据加载完成: 700 条样本, 161 维特征
  标签分布: merged=191, not_merged=509
训练集: 560, 测试集: 140

SVM 评估结果
  Accuracy:   0.8143
  Precision:  0.7143
  Recall:     0.5263
  F1-score:   0.6061
  ROC-AUC:    0.7758
  混淆矩阵:
    TN=94, FP=8
    FN=18, TP=20

Random Forest 评估结果
  Accuracy:   0.8571
  Precision:  0.8750
  Recall:     0.5526
  F1-score:   0.6774
  ROC-AUC:    0.8777
  混淆矩阵:
    TN=99, FP=3
    FN=17, TP=21

特征重要性 Top 10 (Random Forest)
   1. body_len                                           0.0396
   2. cfg_before_call_expression_mean                    0.0274
   3. cfg_after_call_expression_mean                     0.0242
   4. deletions                                          0.0238
   5. additions                                          0.0199
   6. cfg_after_exit_mean                                0.0182
   7. ast_before_for_statement_mean                      0.0178
   8. cfg_before_return_statement_mean                   0.0174
   9. cfg_before_expression_statement_mean        

({'name': 'SVM',
  'accuracy': 0.8142857142857143,
  'precision': 0.7142857142857143,
  'recall': 0.5263157894736842,
  'f1': 0.6060606060606061,
  'roc_auc': 0.7757997936016512,
  'y_test': array([1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1,
         1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1,
         0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
         0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]),
  'y_pred': array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0,
         1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
         0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
         0, 1, 0

---
## 步骤五完成

结果已保存到：
- `data/models/svm_model.joblib` — SVM 模型
- `data/models/rf_model.joblib` — Random Forest 模型
- `data/models/scaler.joblib` — 数据标准化器
- `data/results/confusion_matrix_svm.png` — SVM 混淆矩阵
- `data/results/confusion_matrix_random_forest.png` — RF 混淆矩阵
- `data/results/roc_curve.png` — ROC 曲线
- `data/results/feature_importance_rf.png` — RF 特征重要性
- `data/results/evaluation_report.txt` — 评估报告